# Machine Learning Basics

This notebook covers fundamental machine learning concepts using a dummy dataset.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 2. Create Dummy Dataset

In [ ]:
# Create a dummy dataset for house price prediction
n_samples = 200

data = {
    'Square_Feet': np.random.randint(800, 5000, n_samples),
    'Bedrooms': np.random.randint(1, 6, n_samples),
    'Bathrooms': np.random.randint(1, 4, n_samples),
    'Age_Years': np.random.randint(0, 100, n_samples),
    'Distance_to_City': np.random.randint(1, 50, n_samples)
}

# Create target variable (Price) with some relationship to features
price = (data['Square_Feet'] * 150 + 
         data['Bedrooms'] * 30000 + 
         data['Bathrooms'] * 20000 - 
         data['Age_Years'] * 500 - 
         data['Distance_to_City'] * 2000 + 
         np.random.randn(n_samples) * 50000)

data['Price'] = np.maximum(price, 50000)  # Ensure positive prices

df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset statistics:")
print(df.describe())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Display correlation matrix
print("\nCorrelation Matrix:")
correlation = df.corr()
print(correlation)

In [ ]:
# Visualize correlations
plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize price distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['Price'], bins=30, edgecolor='black', color='skyblue')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.title('Distribution of House Prices')

plt.subplot(1, 2, 2)
plt.scatter(df['Square_Feet'], df['Price'], alpha=0.6, color='green')
plt.xlabel('Square Feet')
plt.ylabel('Price ($)')
plt.title('Price vs Square Feet')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Price', axis=1)
y = df['Price']

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preprocessing complete!")

## 5. Linear Regression Model

In [ ]:
# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_train = lr_model.predict(X_train_scaled)
y_pred_test = lr_model.predict(X_test_scaled)

# Calculate performance metrics
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

print("Linear Regression Results:")
print(f"Training RMSE: ${train_rmse:,.2f}")
print(f"Testing RMSE: ${test_rmse:,.2f}")
print(f"\nModel Coefficients:")
for feature, coef in zip(X.columns, lr_model.coef_):
    print(f"  {feature}: {coef:,.2f}")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_train, alpha=0.6, color='blue', label='Training')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Training Set: Actual vs Predicted')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_test, alpha=0.6, color='green', label='Testing')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Testing Set: Actual vs Predicted')
plt.legend()

plt.tight_layout()
plt.show()

## 6. Classification Example (Price Category Prediction)

In [ ]:
# Create price categories
median_price = df['Price'].median()
df['Price_Category'] = (df['Price'] > median_price).astype(int)

X = df.drop(['Price', 'Price_Category'], axis=1)
y_class = df['Price_Category']

# Split data
X_train, X_test, y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train_class)

# Make predictions
y_pred_class = rf_model.predict(X_test_scaled)

# Evaluate model
accuracy = accuracy_score(y_test_class, y_pred_class)
print(f"Random Forest Accuracy: {accuracy:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test_class, y_pred_class))
print(f"\nClassification Report:")
print(classification_report(y_test_class, y_pred_class, target_names=['Cheap', 'Expensive']))

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='coral')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

print(feature_importance)

## 7. Key Takeaways

- **Data Preprocessing**: Scaling features and splitting data are crucial steps
- **Model Selection**: Different algorithms work better for different problems (regression vs classification)
- **Evaluation Metrics**: RMSE for regression, accuracy/precision/recall for classification
- **Feature Importance**: Understanding which features drive predictions is valuable
- **Overfitting**: Monitor both training and testing performance to detect overfitting